Problem Statement :

When we go grocery shopping, we often have a standard list of things to buy. Each shopper has a distinctive list, depending on one’s needs and preferences. A housewife might buy healthy ingredients for a family dinner, while a bachelor might buy beer and chips. Understanding these buying patterns can help to increase sales in several ways. If there is a pair of items, X and Y, that are frequently bought together:

Both X and Y can be placed on the same shelf, so that buyers of one item would be prompted to buy the other.

Promotional discounts could be applied to just one out of the two items.

Advertisements on X could be targeted at buyers who purchase Y.

X and Y could be combined into a new product, such as having Y in flavors of X.

While we may know that certain items are frequently bought together, the question is, how do we uncover these associations?

Besides increasing sales profits, association rules can also be used in other fields. In medical diagnosis for instance, understanding which symptoms tend to co-morbid can help to improve patient care and medicine prescription.


What is Association Rule Learning?
Association Rule Learning is rule-based learning for identifying the association between different variables in a database. One of the best and most popular examples of Association Rule Learning is the Market Basket Analysis. The problem analyses the association between various items that has the highest probability of being bought together by a customer.

For example, the association rule, {onions, chicken masala} => {chicken} says that a person who has got both onions and chicken masala in his or her basket has a high probability of buying chicken also.

Apriori Algorithm
The algorithm was first proposed in 1994 by Rakesh Agrawal and Ramakrishnan Srikant. Apriori algorithm finds the most frequent itemsets or elements in a transaction database and identifies association rules between the items just like the above-mentioned example.


Image
If you discover that sales of items beyond a certain proportion tend to have a significant impact on your profits, you might consider using that proportion as your support threshold. You may then identify itemsets with support values above this threshold as significant itemsets.

Confidence:

Confidence says how likely item Y is purchased when item X is purchased, expressed as {X -> Y}. This is measured by the proportion of transactions with item X, in which item Y also appears. In Table 1, the confidence of {apple -> beer} is 3 out of 4, or 75%.

Lift is the ratio between the confidence and support.


In [ ]:
#External package need to install
!pip install apyori

In [ ]:
#import all required packages..
import pandas as pd
import numpy as np
from apyori import apriori

In [ ]:
from google.colab import files
ap= files.upload()

In [ ]:
df = pd.read_csv('Market_Basket_Optimisation.csv')

In [ ]:
df.head()

In [ ]:
## Data Cleaning step

# replacing empty value with 0.
df.fillna(0,inplace=True)

In [ ]:
df.head()

In [ ]:
# Data Pre-processing step

# for using aprori , need to convert data in list format..
# transaction = [['apple','almonds'],['apple'],['banana','apple']]....

transactions = []

for i in range(0,len(df)):
    transactions.append([str(df.values[i,j]) for j in range(0,20) if str(df.values[i,j])!='0'])

In [ ]:
## verifying - by printing the 0th transaction
transactions[0]

In [ ]:
## verifying - by printing the 1st transaction
transactions[1]

In [ ]:
# Call apriori function which requires minimum support, confidance and lift, min length is combination of item default is 2".
rules = apriori(transactions, min_support=0.003, min_confidance=0.2, min_lift=3, min_length=2)

## min_support = 0.003 -> means selecting items with min support of 0.3%
## min_confidance = 0.2 -> means min confidance of 20%
## min_lift = 3
## min_length = 2 -> means no. of items in the transaction should be 2

In [ ]:
#it generates a set of rules in a generator file...
rules

In [ ]:
# all rules need to be converted in a list..
Results = list(rules)
Results

In [ ]:
# convert result in a dataframe for further operation...
df_results = pd.DataFrame(Results)

In [ ]:
# as we see "order_statistics" , is itself a list so need to be converted in proper format..
df_results.head()

In [ ]:
# keep support in a separate data frame so we can use later..
support = df_results.support

In [ ]:
'''
convert orderstatistic in a proper format.
order statistic has lhs => rhs as well rhs => lhs
we can choose any one for convience.
Let's choose first one which is 'df_results['ordered_statistics'][i][0]'
'''

#all four empty list which will contain lhs, rhs, confidance and lift respectively.
first_values = []
second_values = []
third_values = []
fourth_value = []

# loop number of rows time and append 1 by 1 value in a separate list..
# first and second element was frozenset which need to be converted in list..
for i in range(df_results.shape[0]):
    single_list = df_results['ordered_statistics'][i][0]
    first_values.append(list(single_list[0]))
    second_values.append(list(single_list[1]))
    third_values.append(single_list[2])
    fourth_value.append(single_list[3])

In [ ]:
# convert all four list into dataframe for further operation..
lhs = pd.DataFrame(first_values)
rhs = pd.DataFrame(second_values)

confidance=pd.DataFrame(third_values,columns=['Confidance'])

lift=pd.DataFrame(fourth_value,columns=['lift'])

In [ ]:
# concat all list together in a single dataframe
df_final = pd.concat([lhs,rhs,support,confidance,lift], axis=1)
df_final

In [ ]:
'''
 we have some of place only 1 item in lhs and some place 3 or more so we need to a proper represenation for User to understand.
 replacing none with ' ' and combining three column's in 1
 example : coffee,none,none is converted to coffee, ,
'''
df_final.fillna(value=' ', inplace=True)
df_final.head()

,lhs,1,rhs,1,2,3,support,confidance,lift
0,brownies,,cottage cheese,,,,0.003467,0.102767,3.238450
1,chicken,,light cream,,,,0.004533,0.075556,4.843305
2,escalope,,mushroom cream sauce,,,,0.005733,0.072269,3.790327
3,escalope,,pasta,,,,0.005867,0.073950,4.700185
4,fresh bread,,tomato juice,,,,0.004267,0.099071,3.273278


In [ ]:
#set column name
#df_final.columns = ['lhs',1,'rhs',2,3,'support','confidance','lift']
#df_final.head()

In [ ]:
# add all three column to lhs itemset only
df_final['lhs'] = df_final['lhs'] + str(", ") + df_final[1]

df_final['rhs'] = df_final['rhs']+str(", ")+df_final[2] + str(", ") + df_final[3]

In [ ]:
df_final.head()

In [ ]:
#drop columns 1,2 and 3 because now we already appended to lhs column.
df_final.drop(columns=[1,2,3],inplace=True)

In [ ]:
#this is final output. You can sort based on the support lift and confidance..
df_final.head()

In [ ]:
## Showing top 10 items, based on lift.  Sorting in desc order
df_final.sort_values('lift', ascending=False).head(10)